In [7]:
import numpy as np
import pandas as pd
from datetime import datetime



def monte_carlo_asian_call(S0, K, T, r, sigma, M=10000):
    """Азиатский колл-опцион: цена + Greeks"""
    
    def price_for_params(S, sigma):
        dt = T / 252
        n_steps = 252
        Z = np.random.randn(M, n_steps)
        paths = np.zeros((M, n_steps + 1))
        paths[:, 0] = S
        
        for t in range(1, n_steps + 1):
            paths[:, t] = paths[:, t-1] * np.exp(
                (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t-1]
            )
        
        avg_prices = np.mean(paths[:, 1:], axis=1)
        payoffs = np.maximum(avg_prices - K, 0)
        return np.exp(-r * T) * np.mean(payoffs)
    
    price = price_for_params(S0, sigma)
    
    eps_S = 0.01 * S0
    price_up_S = price_for_params(S0 + eps_S, sigma)
    price_down_S = price_for_params(S0 - eps_S, sigma)
    delta = (price_up_S - price_down_S) / (2 * eps_S)
    gamma = (price_up_S - 2*price + price_down_S) / (eps_S**2)
    
    eps_sigma = 0.01 * sigma
    price_up_sigma = price_for_params(S0, sigma + eps_sigma)
    price_down_sigma = price_for_params(S0, sigma - eps_sigma)
    vega = (price_up_sigma - price_down_sigma) / (2 * eps_sigma)
    
    return price, delta, gamma, vega

In [8]:
def monte_carlo_asian_put(S0, K, T, r, sigma, M=10000):
    """Азиатский пут-опцион: цена + Greeks"""
    
    def price_for_params(S, sigma):
        dt = T / 252
        n_steps = 252
        Z = np.random.randn(M, n_steps)
        paths = np.zeros((M, n_steps + 1))
        paths[:, 0] = S
        
        for t in range(1, n_steps + 1):
            paths[:, t] = paths[:, t-1] * np.exp(
                (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t-1]
            )
        
        avg_prices = np.mean(paths[:, 1:], axis=1)
        payoffs = np.maximum(K - avg_prices, 0)
        return np.exp(-r * T) * np.mean(payoffs)
    
    price = price_for_params(S0, sigma)
    
    eps_S = 0.01 * S0
    price_up_S = price_for_params(S0 + eps_S, sigma)
    price_down_S = price_for_params(S0 - eps_S, sigma)
    delta = (price_up_S - price_down_S) / (2 * eps_S)
    gamma = (price_up_S - 2*price + price_down_S) / (eps_S**2)
    
    eps_sigma = 0.01 * sigma
    price_up_sigma = price_for_params(S0, sigma + eps_sigma)
    price_down_sigma = price_for_params(S0, sigma - eps_sigma)
    vega = (price_up_sigma - price_down_sigma) / (2 * eps_sigma)
    
    return price, delta, gamma, vega

In [9]:
def monte_carlo_barrier_up_and_out_call(S0, K, T, r, sigma, barrier, M=10000):
    """Барьерный Up-and-Out Call: цена + Greeks"""
    
    def price_for_params(S, sigma):
        dt = T / 252
        n_steps = 252
        Z = np.random.randn(M, n_steps)
        paths = np.zeros((M, n_steps + 1))
        paths[:, 0] = S
        
        for t in range(1, n_steps + 1):
            paths[:, t] = paths[:, t-1] * np.exp(
                (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t-1]
            )
        
        hit_barrier = np.max(paths[:, 1:], axis=1) >= barrier
        final_prices = paths[:, -1]
        payoffs = np.where(hit_barrier, 0, np.maximum(final_prices - K, 0))
        return np.exp(-r * T) * np.mean(payoffs)
    
    price = price_for_params(S0, sigma)
    
    eps_S = 0.01 * S0
    price_up_S = price_for_params(S0 + eps_S, sigma)
    price_down_S = price_for_params(S0 - eps_S, sigma)
    delta = (price_up_S - price_down_S) / (2 * eps_S)
    gamma = (price_up_S - 2*price + price_down_S) / (eps_S**2)
    
    eps_sigma = 0.01 * sigma
    price_up_sigma = price_for_params(S0, sigma + eps_sigma)
    price_down_sigma = price_for_params(S0, sigma - eps_sigma)
    vega = (price_up_sigma - price_down_sigma) / (2 * eps_sigma)
    
    return price, delta, gamma, vega

In [10]:
def generate_asian_dataset(n_samples=100000, n_paths=10000):
    """Генерация датасета азиатских опционов (100,000 примеров)"""
    
    print("Генерация азиатских опционов...")
    start_time = datetime.now()
    data = []
    
    for i in range(n_samples):
        S = np.random.uniform(80, 120)
        K = np.random.uniform(80, 120)
        T = np.random.uniform(0.25, 2.0)
        r = np.random.uniform(0, 0.1)
        sigma = np.random.uniform(0.1, 0.5)
        
        if np.random.choice(['call', 'put']) == 'call':
            price, delta, gamma, vega = monte_carlo_asian_call(S, K, T, r, sigma, M=n_paths)
            opt_type = 1
        else:
            price, delta, gamma, vega = monte_carlo_asian_put(S, K, T, r, sigma, M=n_paths)
            opt_type = 0
        
        data.append([S, K, T, r, sigma, opt_type, price, delta, gamma, vega])
        
        if (i + 1) % 10000 == 0:
            print(f"  {i+1:,} / {n_samples:,} примеров")
    
    df = pd.DataFrame(data, columns=['S', 'K', 'T', 'r', 'sigma', 'option_type', 
                                      'price', 'delta', 'gamma', 'vega'])
    print(f"Завершено за {(datetime.now() - start_time).total_seconds():.1f} сек\n")
    return df

In [11]:
def generate_barrier_dataset(n_samples=100000, n_paths=10000):
    """Генерация датасета барьерных опционов (100,000 примеров)"""
    
    print("Генерация барьерных опционов...")
    start_time = datetime.now()
    data = []
    
    for i in range(n_samples):
        S = np.random.uniform(80, 120)
        K = np.random.uniform(80, 120)
        T = np.random.uniform(0.25, 2.0)
        r = np.random.uniform(0, 0.1)
        sigma = np.random.uniform(0.1, 0.5)
        barrier = np.random.uniform(S + 5, 150)
        
        price, delta, gamma, vega = monte_carlo_barrier_up_and_out_call(
            S, K, T, r, sigma, barrier, M=n_paths
        )
        
        data.append([S, K, T, r, sigma, barrier, price, delta, gamma, vega])
        
        if (i + 1) % 10000 == 0:
            print(f"  {i+1:,} / {n_samples:,} примеров")
    
    df = pd.DataFrame(data, columns=['S', 'K', 'T', 'r', 'sigma', 'barrier',
                                      'price', 'delta', 'gamma', 'vega'])
    print(f"Завершено за {(datetime.now() - start_time).total_seconds():.1f} сек\n")
    return df

In [16]:


    print("=" * 50)
    print("ГЕНЕРАЦИЯ ДАТАСЕТОВ")
    print("=" * 50)
    print(f"Примеров: 100,000")
    print(f"MC траекторий: 10,000")
    print(f"Greeks: Delta, Gamma, Vega")
    print()
    
    # 1. Азиатские опционы (100,000 примеров)
    df_asian = generate_asian_dataset(n_samples=100000, n_paths=10000)
    df_asian.to_csv("asian_options_100k.csv", index=False)
    print("Сохранено: asian_options_100k.csv")
    
    # 2. Барьерные опционы (100,000 примеров)
    df_barrier = generate_barrier_dataset(n_samples=100000, n_paths=10000)
    df_barrier.to_csv("barrier_options_100k.csv", index=False)
    print("Сохранено: barrier_options_100k.csv")
    
    # Статистика
    print("\n" + "=" * 50)
    print("СТАТИСТИКА")
    print("=" * 50)
    print(f"\nАзиатские опционы:")
    print(f"  Цена: mean={df_asian['price'].mean():.4f}, std={df_asian['price'].std():.4f}")
    print(f"  Delta: mean={df_asian['delta'].mean():.4f}, std={df_asian['delta'].std():.4f}")
    print(f"  Gamma: mean={df_asian['gamma'].mean():.4f}, std={df_asian['gamma'].std():.4f}")
    print(f"  Vega:  mean={df_asian['vega'].mean():.4f}, std={df_asian['vega'].std():.4f}")
    
    print(f"\nБарьерные опционы:")
    print(f"  Цена: mean={df_barrier['price'].mean():.4f}, std={df_barrier['price'].std():.4f}")
    print(f"  Delta: mean={df_barrier['delta'].mean():.4f}, std={df_barrier['delta'].std():.4f}")
    print(f"  Gamma: mean={df_barrier['gamma'].mean():.4f}, std={df_barrier['gamma'].std():.4f}")
    print(f"  Vega:  mean={df_barrier['vega'].mean():.4f}, std={df_barrier['vega'].std():.4f}")
    
    print("\n✅ ГОТОВО!")

ГЕНЕРАЦИЯ ДАТАСЕТОВ
Примеров: 100,000
MC траекторий: 10,000
Greeks: Delta, Gamma, Vega

Генерация азиатских опционов...


KeyboardInterrupt: 